### Evaluation for RAG

#### 1. RAG - Data Ingestion, Retriever, Generation

In [ ]:
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import login

login(token=os.getenv("HF_TOKEN"))

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name, device="cpu")

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        vectors = self.model.encode(texts, normalize_embeddings=True)
        return vectors.tolist()

    def embed_query(self, text: str) -> list[float]:
        vector = self.model.encode(text, normalize_embeddings=True)
        return vector.tolist()

In [ ]:
urls = [
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/"
]

docs = [WebBaseLoader(url).load() for url in urls]
doc_list = [sublist[0] for sublist in docs]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=20
)

doc_splits = text_splitter.split_documents(doc_list)
embedding_model = SentenceTransformerEmbeddings()

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embedding_model
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
# retriever.invoke("What is prompt engineering?")

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
from langsmith import traceable

## Add decorator to start tracing
@traceable()
def rag_bot(question: str) -> dict:
    docs = retriever.invoke(question)
    context = " ".join(doc.page_content for doc in docs)
    prompt = f"""
        You are a helpful assistant. Answer the question based on the context given below.
        Question:
        {question}
        Context:
        {context}
    """

    response = llm.invoke([
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ])
    return {
        "answer": response.content,
        "documents": docs
    }


In [ ]:
# res = rag_bot("What are agents?")

#### 2. Test Data -> Question - Answer

In [ ]:
from langsmith import Client
langsmith_client = Client()

examples = [
    {
        "inputs": "According to research by Gekhman et al. (2024) discussed in the blog post on extrinsic hallucinations, how does fine-tuning an LLM on new knowledge affect its propensity to hallucinate?",
        "outputs": "Fine-tuning LLMs on new knowledge increases their tendency to hallucinate. The research showed that models fit examples containing new ('Unknown') knowledge substantially slower than examples consistent with pre-existing knowledge. Once the model eventually fits these new-knowledge training examples, its overall rate of hallucination on evaluation data rises significantly."
    },
    {
        "inputs": "What is the SAFE (Search-Augmented Factuality Evaluator) framework for long-form factuality evaluation, and how does it compute its primary metric?",
        "outputs": "SAFE (Wei et al. 2024) is a long-form factuality evaluation method that uses an LLM agent to iteratively issue Google Search queries and reason about whether external search results support each extracted atomic fact. Its primary evaluation metric is F1@K, which balances factual precision (ratio of supported facts to total facts in the response) with recall (supported facts relative to a maximum target depth K of relevant facts)."
    },
    {
        "inputs": "In the context of white-box adversarial attacks on LLMs, how does GBDA (Gradient-based Distributional Attack) solve the non-differentiable nature of discrete text sampling?",
        "outputs": "GBDA (Guo et al. 2021) uses the Gumbel-Softmax approximation trick. By injecting stochastic Gumbel noise into the categorical token distribution and tuning a temperature parameter (tau), it creates a continuous, differentiable weighted average over token embedding vectors, enabling standard gradient descent optimization of adversarial loss."
    }
]

inputs = [
    {"question": example["inputs"]}
    for example in examples
]
outputs = [
    {"answer": example["outputs"]}
    for example in examples
]
dataset_name = "RAG Test Evaluation"
# dataset = langsmith_client.create_dataset(dataset_name=dataset_name)
dataset_id = "01210fbc-646a-4f51-92a5-bf485dcd4f83"
# langsmith_client.create_examples(
#     dataset_id=dataset_id,
#     inputs=inputs,
#     outputs=outputs
# )

#### 3. Evaluators

##### 1. Correctness

In [ ]:
from typing_extensions import Annotated,TypedDict

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_groq import ChatGroq

grader_llm=ChatGroq(model="qwen/qwen3.8-27b",temperature=0).with_structured_output(CorrectnessGrade,
                                                                         method="json_schema",strict=True)
## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

##### 2. Relevance: Response vs Input

In [ ]:
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0).with_structured_output(RelevanceGrade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

##### 3. Groundedness: Response vs retrieved docs

In [ ]:
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0).with_structured_output(GroundedGrade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

##### 4. Retrieval Relevance

In [ ]:
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

#### Run the evaluation

In [ ]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = langsmith_client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-oss-120b"},
)

experiment_results.to_pandas()